В этом ноутбуке будем собирать информацию по местам привлечения людей по городам, отобранным ранее

In [3]:
import requests
import pandas as pd
import time
import logging

In [4]:
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

In [5]:
logging.basicConfig(
    level=logging.INFO,
    filename="project.log",
    format="%(asctime)s - %(module)s - %(levelname)s - %(funcName)s: %(lineno)d - %(message)s",
    datefmt="%H:%M:%S"
)

logging.info("Логирование настроено")

In [6]:
api_key = 'xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx'
logging.info('получили апи ключ')
cities = pd.read_excel('cities.xlsx')
logging.info(f'получили список из {len(cities)} городов')
cities.head()

,Unnamed: 0,Город,Регион,Федеральный округ,Население,Наличие аэропорта
0,1,Абакан,Хакасия,Сибирский,184284,True
1,26,Анадырь,Чукотский АО,Дальневосточный,13224,True
2,27,Анапа,Краснодарский край,Южный,84804,True
3,28,Ангарск,Иркутская область,Сибирский,216973,False
4,48,Архангельск,Архангельская область,Северо-Западный,294914,True


Проверим, что выдаст нам поиск по ресторанам в первом попавшемся регионе (потом оказалось, что нам это не понадобится)

https://docs.2gis.com/api/search/categories/reference/2.0/catalog/rubric/search#tag/Kategorii

In [7]:
page_rubric = requests.get(
    'https://catalog.api.2gis.com/2.0/catalog/rubric/search',
    params={
        'key': api_key,
        'q': 'рестораны',
        'region_id': '1',
        'page_size': 5
    }
)

In [10]:
print(page_rubric.status_code)
logging.info(f'тестируем работу с апи 2гиса, статус обращения: {page_rubric.status_code}')
print(page_rubric.json())

200
{'meta': {'api_version': '2.0.20097', 'code': 200, 'issue_date': '20260408'}, 'result': {'items': [{'alias': 'restorany', 'branch_count': 244, 'caption': 'Рестораны', 'id': '164', 'keyword': 'Рестораны', 'name': 'Рестораны', 'org_count': 213, 'parent_id': '2', 'title': 'Рестораны', 'type': 'rubric'}, {'alias': 'bystroe_pitanie', 'branch_count': 1881, 'caption': 'Быстрое питание', 'id': '165', 'keyword': 'Быстрое питание', 'name': 'Быстрое питание', 'org_count': 1208, 'parent_id': '2', 'seo_name': 'Кафе / рестораны быстрого питания (фаст-фуды)', 'title': 'Быстрое питание', 'type': 'rubric'}, {'alias': 'karaoke_zaly', 'branch_count': 59, 'caption': 'Караоке', 'id': '21387', 'keyword': 'Караоке-залы', 'name': 'Караоке-залы', 'org_count': 50, 'parent_id': '2', 'seo_name': 'Караоке-клубы / бары', 'title': 'Караоке', 'type': 'rubric'}], 'total': 3}}


Посмотрим, как смотреть на список регионов

https://docs.2gis.com/api/search/regions/reference/2.0/region/list#tag/Regiony

In [11]:
page_region = requests.get(
    'https://catalog.api.2gis.com/2.0/region/list',
    params={
        'key': api_key,
        'page_size': 5
    }
)

In [12]:
print(page_region.status_code)
logging.info(f'тестируем получение списка регионов, статус обращения: {page_region.status_code}')
print(page_region.json())

200
{'meta': {'api_version': '2.0.20097', 'code': 200, 'issue_date': '20260408'}, 'result': {'items': [{'id': '209', 'name': 'Bakı', 'type': 'region'}, {'id': '173', 'name': 'Cyprus', 'type': 'region'}, {'id': '237', 'name': 'Oman', 'type': 'region'}, {'id': '66', 'name': 'Padova', 'type': 'region'}, {'id': '92', 'name': 'Praha', 'type': 'region'}], 'total': 207}}


Вроде все работает, теперь посмотрим на список всех регионов, которые вообще есть, всего их 207, как мы убедились из предыдущего запроса

In [14]:
page_region_all = requests.get(
    'https://catalog.api.2gis.com/2.0/region/list',
    params={
        'key': api_key,
        'page_size': 207
    }
)

In [15]:
data_regions = page_region_all.json()
regions = data_regions['result']['items']
print(len(regions))
logging.info(f'берем список всех регионов из 2гис, получилось {len(regions)} городов')

207


In [16]:
regions

[{'id': '209', 'name': 'Bakı', 'type': 'region'},
 {'id': '173', 'name': 'Cyprus', 'type': 'region'},
 {'id': '237', 'name': 'Oman', 'type': 'region'},
 {'id': '66', 'name': 'Padova', 'type': 'region'},
 {'id': '92', 'name': 'Praha', 'type': 'region'},
 {'id': '216', 'name': 'Riyadh', 'type': 'region'},
 {'id': '101', 'name': 'Santiago', 'type': 'region'},
 {'id': '99', 'name': 'UAE', 'type': 'region'},
 {'id': '69', 'name': 'Абакан', 'type': 'region'},
 {'id': '248', 'name': 'Абхазия', 'type': 'region'},
 {'id': '196', 'name': 'Актау', 'type': 'region'},
 {'id': '167', 'name': 'Актобе', 'type': 'region'},
 {'id': '67', 'name': 'Алматы', 'type': 'region'},
 {'id': '108', 'name': 'Альметьевск', 'type': 'region'},
 {'id': '155', 'name': 'Анадырь', 'type': 'region'},
 {'id': '106', 'name': 'Армавир', 'type': 'region'},
 {'id': '49', 'name': 'Архангельск', 'type': 'region'},
 {'id': '68', 'name': 'Астана', 'type': 'region'},
 {'id': '8', 'name': 'Астрахань', 'type': 'region'},
 {'id': '168

Убедились, что их 207 и посмотрели на структуру выдачи, теперь посмотрим на них в более удобном формате

In [17]:
for region in regions:
    print(region['id'], region['name'])

209 Bakı
173 Cyprus
237 Oman
66 Padova
92 Praha
216 Riyadh
101 Santiago
99 UAE
69 Абакан
248 Абхазия
196 Актау
167 Актобе
67 Алматы
108 Альметьевск
155 Анадырь
106 Армавир
49 Архангельск
68 Астана
8 Астрахань
168 Атырау
176 Баган
121 Балаково
177 Барабинск
4 Барнаул
46 Белгород
20 Бийск
52 Благовещенск
51 Братск
62 Брянск
251 Бухара
77 Великий Новгород
178 Венгерово
25 Владивосток
114 Владикавказ
59 Владимир
33 Волгоград
122 Волгодонск
78 Вологда
31 Воронеж
115 Грозный
179 Довольное
277 Дудинка гп
274 Душанбе
9 Екатеринбург
126 Елец
242 Жезказган
180 Здвинск
65 Иваново
41 Ижевск
11 Иркутск
205 Ишим
70 Йошкар-Ола
89 КавМинВоды
21 Казань
40 Калининград
61 Калуга
109 Каменск-Уральский
127 Камышин
84 Караганда
181 Карасук
278 Караул сп
182 Каргат
5 Кемерово
58 Киров
129 Ковров
207 Когалым
201 Кокшетау
94 Комсомольск-на-Амуре
203 Костанай
34 Кострома
183 Кочки
23 Краснодар
184 Краснозёрское
7 Красноярск
174 Крым
185 Куйбышев
210 Кумертау
186 Купино
10 Курган
73 Курск
240 Кызылорда
112 Кыргы

Теперь проверим, сколько из этих регионов совпадают с нашими городами

In [18]:
regions_dict = {}
for region in regions:
    regions_dict[region['name']] = region['id']

regions_dict

{'Bakı': '209',
 'Cyprus': '173',
 'Oman': '237',
 'Padova': '66',
 'Praha': '92',
 'Riyadh': '216',
 'Santiago': '101',
 'UAE': '99',
 'Абакан': '69',
 'Абхазия': '248',
 'Актау': '196',
 'Актобе': '167',
 'Алматы': '67',
 'Альметьевск': '108',
 'Анадырь': '155',
 'Армавир': '106',
 'Архангельск': '49',
 'Астана': '68',
 'Астрахань': '8',
 'Атырау': '168',
 'Баган': '176',
 'Балаково': '121',
 'Барабинск': '177',
 'Барнаул': '4',
 'Белгород': '46',
 'Бийск': '20',
 'Благовещенск': '52',
 'Братск': '51',
 'Брянск': '62',
 'Бухара': '251',
 'Великий Новгород': '77',
 'Венгерово': '178',
 'Владивосток': '25',
 'Владикавказ': '114',
 'Владимир': '59',
 'Волгоград': '33',
 'Волгодонск': '122',
 'Вологда': '78',
 'Воронеж': '31',
 'Грозный': '115',
 'Довольное': '179',
 'Дудинка гп': '277',
 'Душанбе': '274',
 'Екатеринбург': '9',
 'Елец': '126',
 'Жезказган': '242',
 'Здвинск': '180',
 'Иваново': '65',
 'Ижевск': '41',
 'Иркутск': '11',
 'Ишим': '205',
 'Йошкар-Ола': '70',
 'КавМинВоды': '

In [19]:
found = {}
not_found = []

for city in cities['Город']:
    if city in regions_dict:
        found[city] = regions_dict[city]
    else:
        not_found.append(city)

print(len(found))
print(len(not_found))

79
25


То есть, есть 25 городов, которых не нашлось в 2ГИС, посмотрим, что это за города

In [20]:
not_found

['Анапа',
 'Ангарск',
 'Балашиха',
 'Великие Луки',
 'Волжский',
 'Горно-Алтайск',
 'Дзержинск',
 'Жуковский',
 'Златоуст',
 'Королёв',
 'Красногорск',
 'Люберцы',
 'Магас',
 'Миасс',
 'Минеральные Воды',
 'Мурманск',
 'Мытищи',
 'Нижнекамск',
 'Одинцово',
 'Подольск',
 'Северодвинск',
 'Симферополь',
 'Химки',
 'Череповец',
 'Энгельс']

Я посмотрел в наши списки, к сожалению, большинства городов реально у нас нет, но, есть 3 интересные вещи:

1) Миасс и Златоуст есть в обоих наших списках, но в 2ГИСе они собраны в 1 единый регион, буквально: "Миасс и Златоуст", поэтому их добавим вручную

2) В 2ГИСе нет Мурманска, но есть Мурманская область, но Мурманск является крупнейшим городом своей области, поэтому тоже добавим

3) В 2ГИСе нет многих городов мос области, которые есть в нашем списке городов, но есть единая "Московская область". Однако, так как там все же много городов, добавлять мы ее не будем

In [21]:
found['Златоуст'] = '87'
found['Миасс'] = '87'
found['Мурманск'] = '96'

print(len(found))
print(len(not_found) - 3)
print((len(found) / cities.shape[0]) * 100)
logging.info(f'посмотрели частичные и полные совпадения по городам из нашего файла и из 2гиса, покрываем {(len(found) / cities.shape[0]) * 100}% наших городов')

82
22
78.84615384615384


Почти 79% наших городов будут в 2ГИС

Теперь посмотрим, какие конкретно рубрики нам нужны, проверим, какие категории рубрик вообще существуют

https://docs.2gis.com/api/search/categories/reference/2.0/catalog/rubric/list#tag/Kategorii

In [23]:
rubric_categories_response = requests.get(
    'https://catalog.api.2gis.com/2.0/catalog/rubric/list',
    params={
        'key': api_key,
        'region_id': '1',
        'page_size': 50
    }
)

rubric_cat_data = rubric_categories_response.json()
rubric_cat_items = rubric_cat_data['result']['items']

print(rubric_cat_data['result']['total']) # меньше 50, значит все вывели
for item in rubric_cat_items:
    print(f'id: {item['id']}, название: {item['name']}')
logging.info('смотрим, какие конкретно виды мест притяжения людей нас заинтересуют')

28
id: 5419, название: Аварийные / справочные / экстренные службы
id: 42903, название: Автосервис / Автотовары
id: 1, название: Город / Власть
id: 2, название: Досуг / Развлечения / Общественное питание
id: 19532, название: Интернет / Связь / Информационные технологии
id: 10, название: Коммунальные / бытовые / ритуальные услуги
id: 3, название: Компьютеры / Бытовая техника / Офисная техника
id: 4, название: Культура / Искусство / Религия
id: 787, название: Мебель / Материалы / Фурнитура
id: 5, название: Медицина / Здоровье / Красота
id: 112663, название: Места
id: 13, название: Металлы / Топливо / Химия
id: 15, название: Оборудование / Инструмент
id: 6, название: Образование / Работа / Карьера
id: 1035, название: Одежда / Обувь
id: 1621, название: Охрана / Безопасность
id: 14, название: Продукты питания / Напитки
id: 7, название: Реклама / Полиграфия / СМИ
id: 8, название: Спорт / Отдых / Туризм
id: 42867, название: Строительные / отделочные материалы
id: 9, название: Строительство / Н

Меня здесь привлекло 4 категории: 2 (Досуг / Развлечения / Общественное питание), 4 (Культура / Искусство / Религия), 112663 (Места) и 8 (Спорт / Отдых / Туризм)

Кажется, именно в этих категориях будет то, что влияет на привлекательность города для туриста

Посмотрим глубже в каждую из категорий, начнем со второй

In [24]:
rubric_sub_response = requests.get(
    'https://catalog.api.2gis.com/2.0/catalog/rubric/list',
    params={
        'key': api_key,
        'region_id': '1',
        'parent_id': '2',
        'page_size': 50
    }
)

rubric_sub_data = rubric_sub_response.json()
rubric_sub_items = rubric_sub_data['result']['items']

for item in rubric_sub_items:
    print(f'id: {item['id']}, название: {item['name']}')

id: 112455, название: Авиатренажёры
id: 113336, название: Автоматы фотопечати
id: 537, название: Аквапарки
id: 72370, название: Антикафе
id: 110305, название: Аренда беседок
id: 112792, название: Аренда детских лофтов
id: 110373, название: Аренда мебели для мероприятий
id: 110308, название: Аренда шатров и конструкций
id: 110358, название: Аттракционы
id: 100772, название: Аэродинамические комплексы
id: 110384, название: Аэросъёмка
id: 946, название: Бани и сауны
id: 10803, название: Банкетные залы
id: 159, название: Бары
id: 21497, название: Билеты на мероприятия
id: 169, название: Бильярдные залы
id: 24169, название: Ботанический сад
id: 170, название: Боулинг
id: 171, название: Букмекеры
id: 165, название: Быстрое питание
id: 110304, название: Верёвочные парки
id: 24099, название: Выставка насекомых
id: 8151, название: Дайвинг-клубы
id: 9508, название: Дельфинарий
id: 16743, название: Детские игровые залы
id: 113330, название: Детские развлекательные автоматы
id: 13787, название: До

Отсюда мы возьмем рестораны (164)

Посморим остальные категории

In [25]:
for cat in ['4', '112663', '8']:
    rubric_sub_response = requests.get(
        'https://catalog.api.2gis.com/2.0/catalog/rubric/list',
        params={
            'key': api_key,
            'region_id': '1',
            'parent_id': cat,
            'page_size': 100
        }
    )
    rubric_sub_items = rubric_sub_response.json()['result']['items']
    print(f'------ {cat} ------')
    for item in rubric_sub_items:
        print(f'id: {item['id']}, название: {item['name']}')

    time.sleep(0.5)

------ 4 ------
id: 686, название: Антиквариат
id: 112395, название: Багетные мастерские
id: 189, название: Библиотеки
id: 18574, название: Воскресные школы
id: 112657, название: Гончарные мастерские
id: 112875, название: Городские мероприятия
id: 10887, название: Городские оркестры
id: 54914, название: Дацаны
id: 51206, название: Духовные учебные заведения
id: 51323, название: Иконописные мастерские
id: 113326, название: Интерактивные музеи
id: 191, название: Киностудии
id: 13374, название: Мечети
id: 19575, название: Монастыри
id: 193, название: Музеи
id: 110420, название: Нумизматика
id: 52186, название: Паломнические поездки
id: 11647, название: Планетарий
id: 110361, название: Подвесы и освещение картин
id: 19576, название: Приходы
id: 1175, название: Религиозные организации
id: 12249, название: Религиозные товары
id: 51379, название: Реставрация исторических памятников
id: 54063, название: Синагога
id: 199, название: Театры
id: 200, название: Филармония
id: 113154, название: Фила

Из этих категорий возьмем следующее: музеи (193), театры (199), природные достопримечательности (112668) и интересные здания (112670)

Итого, список такой:

In [26]:
rubrics = {
    'Рестораны': '164',
    'Музеи': '193',
    'Театры': '199',
    'Природные достопримечательности': '112668',
    'Интересные здания': '112670'
}
logging.info(f'выбрали конкретные виды мест (рубрики), это {[rub for rub in rubrics]}')

Сейчас посмотрим на примере одного города (чтобы не тратить сразу много запросов, а то лимит 1000), что у нас для него будет, возьмем самый большой город - Москву

In [27]:
for city in found:
  if city == 'Москва':
    print(city, found[city])
    break

Москва 32


https://docs.2gis.com/api/search/places/reference/3.0/items/#tag/Katalog/paths/~13.0~1items/get

In [28]:
moscow_response = requests.get(
    'https://catalog.api.2gis.com/3.0/items',
    params={
        'key': api_key,
        'rubric_id': '164',
        'region_id': '32',
        'page_size': 10, # больше нельзя за 1 запрос, к сожалению (причем в документации написано, что можно 50, но у меня выдавало ошибку при числах больше 10)
        'fields': 'items.point,items.reviews',
        'sort': 'rating' # возьмем самые популярные места
    }
)

moscow_data = moscow_response.json()
print(moscow_data)
moscow_items = moscow_data['result']['items']

print(len(moscow_items), moscow_data['result']['total'])
logging.info('посмотрели на примере Москвы, как будем обращаться ко всем городам')

{'meta': {'api_version': '3.0.20097', 'code': 200, 'issue_date': '20260408'}, 'result': {'items': [{'address_comment': '2 этаж', 'address_name': 'Большая Садовая улица, 5 к1', 'id': '70000001097484511', 'name': 'Infinitum, лаундж-бар', 'point': {'lat': 55.768619, 'lon': 37.592217}, 'reviews': {'general_rating': 5, 'general_review_count': 312, 'general_review_count_with_stars': 314, 'is_reviewable': True, 'is_reviewable_on_flamp': True, 'items': [{'is_reviewable': True, 'tag': '2gis_reviews'}, {'is_reviewable': True, 'tag': 'flamp'}], 'org_rating': 5, 'org_review_count': 512, 'org_review_count_with_stars': 515}, 'type': 'branch'}, {'address_name': 'улица Солянка, 1/2 ст2', 'id': '70000001105182528', 'name': 'Виновные, винный бар', 'point': {'lat': 55.753816, 'lon': 37.638522}, 'reviews': {'general_rating': 5, 'general_review_count': 173, 'general_review_count_with_stars': 176, 'is_reviewable': True, 'is_reviewable_on_flamp': True, 'items': [{'is_reviewable': True, 'tag': '2gis_reviews'}

Пробуем для всех наших городов найти места

Тут сразу уточню, что этот код я переделывал несколько раз. Код в том виде, который написан тут, по идее, должен работать, но в первый раз была ошибка, что general_rating был не везде

На втором запуске спустя ~355 запросов у какого-то элемента стало отсутствовать поле result, поэтому был добавлен блок, который учитывает такие проблемы + стало интересно посмотреть, где ошибка

In [ ]:
all_places = []

for city, city_id in found.items():
  for rubric, rubric_id in rubrics.items():
    response = requests.get(
        'https://catalog.api.2gis.com/3.0/items',
        params={
            'key': api_key,
            'rubric_id': rubric_id,
            'region_id': city_id,
            'page_size': 10,
            'fields': 'items.point,items.reviews',
            'sort': 'rating'
        }
    )
    data = response.json()
    if 'result' not in data: # пришлось пожертвовать 35% лимита запросов, чтобы понять, что этот блок нужен
              print(f'Ошибка: {city} / {rubric}: {data}')
              continue
    ditems = data['result']['items']

    for place in ditems:
      if 'name' in place:
        name = place['name']
      else:
        name = ''

      if 'point' in place:
        lat = place['point']['lat'] if 'lat' in place['point'] else None
        lon = place['point']['lon'] if 'lon' in place['point'] else None
      else:
        lat = None
        lon = None

      if 'reviews' in place:
        rating = place['reviews']['general_rating'] if 'general_rating' in place['reviews'] else None
        review_count = place['reviews']['general_review_count'] if 'general_review_count' in place['reviews'] else 0
      else:
        rating = None
        review_count = 0

      all_places.append({
          'city': city,
          'rubric': rubric,
          'name': name,
          'lat': lat,
          'lon': lon,
          'rating': rating,
          'review_count': review_count
      })
    time.sleep(0.5)

  pd.DataFrame(all_places).to_csv('places_2gis.csv', index=False, encoding='utf-8-sig')

Ошибка: Хасавюрт / Интересные здания: {'meta': {'api_version': '3.0.20097', 'code': 404, 'error': {'message': 'Results not found', 'type': 'itemNotFound'}, 'issue_date': '20260407'}}


In [29]:
logging.info('попробовали сделать перебор всех интересных мест по всем городам, но возникла ошибка')

Как будет видно до следующего запуска похожего кода, здесь получилось что-то не то. Я немного напутал с отступами и у меня перебиралась только одна категория

In [ ]:
print(all_places)

[{'city': 'Абакан', 'rubric': 'Интересные здания', 'name': 'Тубтен Шедруб Линг, буддийский монастырь', 'lat': 51.691496, 'lon': 94.404949, 'rating': 5, 'review_count': 33}, {'city': 'Абакан', 'rubric': 'Интересные здания', 'name': 'Краеведческий музей им. Н.М. Мартьянова', 'lat': 53.709695, 'lon': 91.688606, 'rating': 4.9, 'review_count': 80}, {'city': 'Абакан', 'rubric': 'Интересные здания', 'name': 'Хурээ Цеченлинг, буддийский храм', 'lat': 51.723753, 'lon': 94.450489, 'rating': 5, 'review_count': 15}, {'city': 'Абакан', 'rubric': 'Интересные здания', 'name': 'Октябрьская улица, 65', 'lat': 53.709955, 'lon': 91.69699, 'rating': 5, 'review_count': 14}, {'city': 'Абакан', 'rubric': 'Интересные здания', 'name': 'Драматический театр', 'lat': 53.709711, 'lon': 91.683435, 'rating': 4.9, 'review_count': 43}, {'city': 'Абакан', 'rubric': 'Интересные здания', 'name': 'Саяно-Шушенская ГЭС им. П.С. Непорожнего', 'lat': 52.829654, 'lon': 91.368377, 'rating': 4.9, 'review_count': 64}, {'city': 'А

In [ ]:
df_2gis = pd.DataFrame(all_places)
df_2gis.to_csv('df_2gis', index=False, encoding='utf-8-sig')

In [ ]:
df_2gis.head(20)

,city,rubric,name,lat,lon,rating,review_count
0,Абакан,Интересные здания,"Тубтен Шедруб Линг, буддийский монастырь",51.691496,94.404949,5.0,33
1,Абакан,Интересные здания,Краеведческий музей им. Н.М. Мартьянова,53.709695,91.688606,4.9,80
2,Абакан,Интересные здания,"Хурээ Цеченлинг, буддийский храм",51.723753,94.450489,5.0,15
3,Абакан,Интересные здания,"Октябрьская улица, 65",53.709955,91.696990,5.0,14
4,Абакан,Интересные здания,Драматический театр,53.709711,91.683435,4.9,43
5,Абакан,Интересные здания,Саяно-Шушенская ГЭС им. П.С. Непорожнего,52.829654,91.368377,4.9,64
6,Абакан,Интересные здания,Собор Николая Чудотворца,53.710005,91.479213,5.0,10
7,Абакан,Интересные здания,Национальный центр народного творчества им. С....,53.724394,91.448497,4.8,23
8,Абакан,Интересные здания,Краеведческий музей им. Л.Р. Кызласова,53.720434,91.473981,4.8,203
9,Абакан,Интересные здания,Тувгосфилармония им. В.М. Халилова,51.722006,94.433917,5.0,4


In [ ]:
print(len(all_places))

798


In [ ]:
print(df_2gis['rubric'].value_counts())

rubric
Интересные здания    798
Name: count, dtype: int64


In [30]:
logging.info('разобрались, в чем ошибка (в отступах), поменяли ключ и начинаем уже реально брать данные о всех интересных местах по всем городам')

Из-за такой ситуации израсходовалось 780/1000 запросов

Поэтому пробуем новый ключ

In [31]:
api_key_new = 'yyyyyyyy-yyyy-yyyy-yyyy-yyyyyyyyyyyy'

In [32]:
logging.info('начинаем сбор, для начала возьмем первую страницу')
all_places = []

for city, city_id in found.items():
  for rubric, rubric_id in rubrics.items():
    response = requests.get(
        'https://catalog.api.2gis.com/3.0/items',
        params={
            'key': api_key_new,
            'rubric_id': rubric_id,
            'region_id': city_id,
            'page': 1,
            'page_size': 10,
            'fields': 'items.point,items.reviews',
            'sort': 'rating'
        }
    )
    data = response.json()
    if 'result' not in data:
      logging.warning(f'Ошибка: {city} / {rubric}: {data}')
      print(f'Ошибка: {city} / {rubric}: {data}')
      continue
    ditems = data['result']['items']

    for place in ditems:
      if 'name' in place:
        name = place['name']
      else:
        name = ''

      if 'point' in place:
        lat = place['point']['lat'] if 'lat' in place['point'] else None
        lon = place['point']['lon'] if 'lon' in place['point'] else None
      else:
        lat = None
        lon = None

      if 'reviews' in place:
        rating = place['reviews']['general_rating'] if 'general_rating' in place['reviews'] else None
        review_count = place['reviews']['general_review_count'] if 'general_review_count' in place['reviews'] else 0
      else:
        rating = None
        review_count = 0

      all_places.append({
          'city': city,
          'rubric': rubric,
          'name': name,
          'lat': lat,
          'lon': lon,
          'rating': rating,
          'review_count': review_count
      })
    time.sleep(0.5)

  pd.DataFrame(all_places).to_csv('places_2gis_1.csv', index=False, encoding='utf-8-sig')

Ошибка: Анадырь / Театры: {'meta': {'api_version': '3.0.20097', 'code': 404, 'error': {'message': 'Results not found', 'type': 'itemNotFound'}, 'issue_date': '20260408'}}
Ошибка: Грозный / Природные достопримечательности: {'meta': {'api_version': '3.0.20097', 'code': 404, 'error': {'message': 'Results not found', 'type': 'itemNotFound'}, 'issue_date': '20260408'}}
Ошибка: Орск / Природные достопримечательности: {'meta': {'api_version': '3.0.20097', 'code': 404, 'error': {'message': 'Results not found', 'type': 'itemNotFound'}, 'issue_date': '20260408'}}
Ошибка: Рыбинск / Природные достопримечательности: {'meta': {'api_version': '3.0.20097', 'code': 404, 'error': {'message': 'Results not found', 'type': 'itemNotFound'}, 'issue_date': '20260408'}}
Ошибка: Сыктывкар / Природные достопримечательности: {'meta': {'api_version': '3.0.20097', 'code': 404, 'error': {'message': 'Results not found', 'type': 'itemNotFound'}, 'issue_date': '20260408'}}
Ошибка: Хасавюрт / Театры: {'meta': {'api_vers

In [33]:
logging.info(f'сбор данных по первым страницам завершен, собрали и сохранили {len(all_places)} мест')

In [34]:
pd.DataFrame(all_places)['rubric'].value_counts()

,count
rubric,
Рестораны,810
Интересные здания,798
Музеи,757
Природные достопримечательности,563
Театры,548


В данных все норм, код пофиксили

Ошибки дальше выводить не буду, нет смысла, но оставлю в логах

Так как у нас свежий апи ключ, сделаем сразу вторую страницу, чтобы было больше данных

In [35]:
logging.info('начинаем сбор вторых страниц')
all_places2 = []

for city, city_id in found.items():
  for rubric, rubric_id in rubrics.items():
    response = requests.get(
        'https://catalog.api.2gis.com/3.0/items',
        params={
            'key': api_key_new,
            'rubric_id': rubric_id,
            'region_id': city_id,
            'page': 2,
            'page_size': 10,
            'fields': 'items.point,items.reviews',
            'sort': 'rating'
        }
    )
    data = response.json()
    if 'result' not in data:
      logging.warning(f'Ошибка: {city} / {rubric}: {data}')
      continue
    ditems = data['result']['items']

    for place in ditems:
      if 'name' in place:
        name = place['name']
      else:
        name = ''

      if 'point' in place:
        lat = place['point']['lat'] if 'lat' in place['point'] else None
        lon = place['point']['lon'] if 'lon' in place['point'] else None
      else:
        lat = None
        lon = None

      if 'reviews' in place:
        rating = place['reviews']['general_rating'] if 'general_rating' in place['reviews'] else None
        review_count = place['reviews']['general_review_count'] if 'general_review_count' in place['reviews'] else 0
      else:
        rating = None
        review_count = 0

      all_places2.append({
          'city': city,
          'rubric': rubric,
          'name': name,
          'lat': lat,
          'lon': lon,
          'rating': rating,
          'review_count': review_count
      })
    time.sleep(0.5)

  pd.DataFrame(all_places2).to_csv('places_2gis_2.csv', index=False, encoding='utf-8-sig')

In [36]:
logging.info(f'сбор данных по вторым страницам завершен, собрали и сохранили {len(all_places2)} мест')

In [37]:
pd.DataFrame(all_places2)['rubric'].value_counts()

,count
rubric,
Интересные здания,748
Рестораны,744
Музеи,604
Природные достопримечательности,390
Театры,191


Тут появился еще один апи ключ...

Поэтому сделаем 3 и 4 страницы

In [38]:
the_newest_key = 'zzzzzzzz-zzzz-zzzz-zzzz-zzzzzzzzzz'

In [39]:
logging.info('взяли новый ключ, собираем данные третьих страниц')
all_places3_new = []

for city, city_id in found.items():
  for rubric, rubric_id in rubrics.items():
    response = requests.get(
        'https://catalog.api.2gis.com/3.0/items',
        params={
            'key': the_newest_key,
            'rubric_id': rubric_id,
            'region_id': city_id,
            'page': 3,
            'page_size': 10,
            'fields': 'items.point,items.reviews',
            'sort': 'rating'
        }
    )
    data = response.json()
    if 'result' not in data:
      logging.warning(f'Ошибка: {city} / {rubric}: {data}')
      continue
    ditems = data['result']['items']

    for place in ditems:
      if 'name' in place:
        name = place['name']
      else:
        name = ''

      if 'point' in place:
        lat = place['point']['lat'] if 'lat' in place['point'] else None
        lon = place['point']['lon'] if 'lon' in place['point'] else None
      else:
        lat = None
        lon = None

      if 'reviews' in place:
        rating = place['reviews']['general_rating'] if 'general_rating' in place['reviews'] else None
        review_count = place['reviews']['general_review_count'] if 'general_review_count' in place['reviews'] else 0
      else:
        rating = None
        review_count = 0

      all_places3_new.append({
          'city': city,
          'rubric': rubric,
          'name': name,
          'lat': lat,
          'lon': lon,
          'rating': rating,
          'review_count': review_count
      })
    time.sleep(0.5)

  pd.DataFrame(all_places3_new).to_csv('places_2gis_3_new.csv', index=False, encoding='utf-8-sig') # не 3, а 3_new, потому что сначала забыл ключ другой поставить

In [40]:
logging.info(f'сбор данных по третьим страницам завершен, собрали и сохранили {len(all_places3_new)} мест')

In [41]:
pd.DataFrame(all_places3_new)['rubric'].value_counts()

,count
rubric,
Интересные здания,709
Рестораны,678
Музеи,498
Природные достопримечательности,301
Театры,87


In [42]:
logging.info('наконец, собираем четвертые страницы')
all_places4 = []

for city, city_id in found.items():
  for rubric, rubric_id in rubrics.items():
    response = requests.get(
        'https://catalog.api.2gis.com/3.0/items',
        params={
            'key': the_newest_key,
            'rubric_id': rubric_id,
            'region_id': city_id,
            'page': 4,
            'page_size': 10,
            'fields': 'items.point,items.reviews',
            'sort': 'rating'
        }
    )
    data = response.json()
    if 'result' not in data:
      logging.warning(f'Ошибка: {city} / {rubric}: {data}')
      continue
    ditems = data['result']['items']

    for place in ditems:
      if 'name' in place:
        name = place['name']
      else:
        name = ''

      if 'point' in place:
        lat = place['point']['lat'] if 'lat' in place['point'] else None
        lon = place['point']['lon'] if 'lon' in place['point'] else None
      else:
        lat = None
        lon = None

      if 'reviews' in place:
        rating = place['reviews']['general_rating'] if 'general_rating' in place['reviews'] else None
        review_count = place['reviews']['general_review_count'] if 'general_review_count' in place['reviews'] else 0
      else:
        rating = None
        review_count = 0

      all_places4.append({
          'city': city,
          'rubric': rubric,
          'name': name,
          'lat': lat,
          'lon': lon,
          'rating': rating,
          'review_count': review_count
      })
    time.sleep(0.5)

  pd.DataFrame(all_places4).to_csv('places_2gis_4.csv', index=False, encoding='utf-8-sig')

In [43]:
logging.info(f'сбор данных по четвертым страницам завершен, собрали и сохранили {len(all_places3_new)} мест')

In [44]:
pd.DataFrame(all_places4)['rubric'].value_counts()

,count
rubric,
Интересные здания,651
Рестораны,631
Музеи,406
Природные достопримечательности,232
Театры,37


Все 4 страницы сделаны, теперь соберем все воедино

In [45]:
df1 = pd.read_csv('places_2gis_1.csv', encoding='utf-8-sig')
df2 = pd.read_csv('places_2gis_2.csv', encoding='utf-8-sig')
df3 = pd.read_csv('places_2gis_3_new.csv', encoding='utf-8-sig')
df4 = pd.read_csv('places_2gis_4.csv', encoding='utf-8-sig')

df_all = pd.concat([df1, df2, df3, df4], ignore_index=True)
logging.info(f'собрали все места в единый датасет, в нем {df_all.shape[0]} наблюдений')

Ну и посмотрим немного статистики по получившемуся датафрейму + сохраним его

In [46]:
df_all.shape

(10383, 7)

In [47]:
df_all.head()

,city,rubric,name,lat,lon,rating,review_count
0,Абакан,Рестораны,Лисья нора,52.849878,91.420183,4.9,401
1,Абакан,Рестораны,"STIX, бельгийский ресторан",53.721736,91.444138,4.7,648
2,Абакан,Рестораны,"Мажор, караоке-бар",53.725752,91.456086,4.7,309
3,Абакан,Рестораны,"Мадрид, кафе",53.727392,91.437827,4.6,300
4,Абакан,Рестораны,"Баран и бисер, ресторан",53.724203,91.443411,4.6,647


In [48]:
df_all['rubric'].value_counts()

,count
rubric,
Интересные здания,2906
Рестораны,2863
Музеи,2265
Природные достопримечательности,1486
Театры,863


In [49]:
df_all['city'].value_counts()

,count
city,
Екатеринбург,200
Москва,200
Санкт-Петербург,200
Новосибирск,194
Красноярск,193
...,...
Магадан,35
Элиста,33
Шахты,29


In [50]:
df_all.to_csv('places_2gis_all.csv', index=False, encoding='utf-8-sig')